# TrainData Inspection
Run all cells and paste the output back — used to confirm year splits and folder structure before running nb_06.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import re
from collections import defaultdict

ROOT = Path("/content/drive/MyDrive/MLfTCC/Data/Tropicyclonenet/TrainData")
BASINS = ["EP", "NA", "NI", "SI", "SP", "WP"]

def parse_year(name):
    m = re.search(r"(\d{4})", name)
    return int(m.group(1)) if m else None

print(f"Root exists: {ROOT.exists()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Root exists: True


In [2]:
# ── 1. Top-level contents ──────────────────────────────────────────────────
print("[1] Top-level contents of TrainData/")
print("-" * 50)
for p in sorted(ROOT.iterdir()):
    kind = "DIR " if p.is_dir() else "FILE"
    print(f"  {kind}  {p.name}")

[1] Top-level contents of TrainData/
--------------------------------------------------
  DIR   Data1D
  DIR   EP
  DIR   Env-Data
  DIR   NA
  DIR   NI
  DIR   SI
  DIR   SP
  DIR   WP


In [3]:
# ── 2. Data1D subdir structure ─────────────────────────────────────────────
data1d = ROOT / "Data1D"
print(f"[2] Data1D exists: {data1d.exists()}")
print("-" * 50)
if data1d.exists():
    for d in sorted(data1d.iterdir()):
        if d.is_dir():
            subdirs = [x.name for x in sorted(d.iterdir()) if x.is_dir()]
            txt_count = len(list(d.rglob("*.txt")))
            print(f"  Data1D/{d.name}/")
            print(f"    subdirs : {subdirs if subdirs else '(none — flat)'}")
            print(f"    .txt files: {txt_count}")

[2] Data1D exists: True
--------------------------------------------------
  Data1D/EP/
    subdirs : ['test', 'train', 'val']
    .txt files: 497
  Data1D/NA/
    subdirs : ['test', 'train', 'val']
    .txt files: 517
  Data1D/NI/
    subdirs : ['test', 'train', 'val']
    .txt files: 48
  Data1D/SI/
    subdirs : ['test', 'train', 'val']
    .txt files: 557
  Data1D/SP/
    subdirs : ['test', 'train', 'val']
    .txt files: 308
  Data1D/WP/
    subdirs : ['test', 'train', 'val']
    .txt files: 1799


In [4]:
# ── 3. Per-basin year range ────────────────────────────────────────────────
print("[3] Per-basin year range (from filenames)")
print("-" * 60)
grand_years = set()

for basin in BASINS:
    basin_dir = data1d / basin
    if not basin_dir.exists():
        print(f"  {basin}: NOT FOUND under Data1D")
        continue

    files = list(basin_dir.rglob("*.txt"))
    years = sorted(set(y for f in files if (y := parse_year(f.name))))

    if years:
        grand_years.update(years)
        full_range = set(range(min(years), max(years)+1))
        missing = sorted(full_range - set(years))
        print(f"  {basin}: {min(years)} – {max(years)}  "
              f"({len(years)} distinct years, {len(files)} files)")
        if missing:
            print(f"    gap years: {missing}")
    else:
        print(f"  {basin}: no .txt files found")

if grand_years:
    print(f"\n  ► Overall: {min(grand_years)} – {max(grand_years)} across all basins")

[3] Per-basin year range (from filenames)
------------------------------------------------------------
  EP: 1988 – 2023  (36 distinct years, 497 files)
  NA: 1960 – 2023  (64 distinct years, 517 files)
  NI: 1992 – 2022  (20 distinct years, 48 files)
    gap years: [1993, 1994, 1995, 1996, 1998, 1999, 2000, 2001, 2002, 2003, 2004]
  SI: 1973 – 2023  (51 distinct years, 557 files)
  SP: 1970 – 2023  (53 distinct years, 308 files)
    gap years: [1971]
  WP: 1950 – 2023  (74 distinct years, 1799 files)

  ► Overall: 1950 – 2023 across all basins


In [5]:
# ── 4. Sample filenames ────────────────────────────────────────────────────
print("[4] Sample filenames (first 5 per basin)")
print("-" * 60)
for basin in BASINS:
    basin_dir = data1d / basin
    if not basin_dir.exists():
        continue
    files = sorted(basin_dir.rglob("*.txt"))[:5]
    for f in files:
        print(f"  {f.relative_to(data1d)}")
    if files:
        print()

[4] Sample filenames (first 5 per basin)
------------------------------------------------------------
  EP/test/EP2017BSTDORA.txt
  EP/test/EP2017BSTEUGENE.txt
  EP/test/EP2017BSTFERNANDA.txt
  EP/test/EP2017BSTGREG.txt
  EP/test/EP2017BSTHILARY.txt

  NA/test/NA2017BSTARLENE.txt
  NA/test/NA2017BSTCINDY.txt
  NA/test/NA2017BSTFRANKLIN.txt
  NA/test/NA2017BSTGERT.txt
  NA/test/NA2017BSTHARVEY.txt

  NI/test/NI2017BSTOCKHI.txt
  NI/test/NI2018BSTGAJA.txt
  NI/test/NI2018BSTLUBAN.txt
  NI/test/NI2018BSTMEKUNU-SAGAR.txt
  NI/test/NI2018BSTPHETHAI.txt

  SI/test/SI2017BSTABELA.txt
  SI/test/SI2017BSTBLANCHE.txt
  SI/test/SI2017BSTBRANSBY.txt
  SI/test/SI2017BSTCALEB.txt
  SI/test/SI2017BSTCARLOS.txt

  SP/test/SP2017BSTALFRED.txt
  SP/test/SP2017BSTCOOK.txt
  SP/test/SP2017BSTDEBBIE.txt
  SP/test/SP2017BSTDONNA.txt
  SP/test/SP2017BSTELLA.txt

  WP/test/WP2017BSTBANYAN.txt
  WP/test/WP2017BSTDAMREY.txt
  WP/test/WP2017BSTDOKSURI.txt
  WP/test/WP2017BSTGUCHOL.txt
  WP/test/WP2017BSTHAIKUI.t

In [6]:
# ── 5. Peek inside one file — confirm column format ────────────────────────
print("[5] First file content (first 5 rows)")
print("-" * 60)
for basin in BASINS:
    basin_dir = data1d / basin
    if not basin_dir.exists():
        continue
    files = sorted(basin_dir.rglob("*.txt"))
    if not files:
        continue
    sample_file = files[0]
    print(f"  File: {sample_file.relative_to(data1d)}")
    with open(sample_file) as fh:
        for i, line in enumerate(fh):
            if i >= 5: break
            print(f"    {line.rstrip()}")
    print()
    break   # one file is enough to confirm format

[5] First file content (first 5 rows)
------------------------------------------------------------
  File: EP/test/EP2017BSTDORA.txt
    0.0	1.0	16.700	2.560	0.960	-1.086	2017062400	DORA
    1.0	1.0	16.560	2.600	0.960	-1.086	2017062406	DORA
    2.0	1.0	16.420	2.640	0.960	-1.086	2017062412	DORA
    3.0	1.0	16.260	2.720	0.940	-1.086	2017062418	DORA
    4.0	1.0	16.100	2.800	0.920	-0.983	2017062500	DORA



In [7]:
# ── 6. Env-Data and Data3D folders ─────────────────────────────────────────
print("[6] Env-Data")
print("-" * 50)
env = ROOT / "Env-Data"
if env.exists():
    print(f"  Env-Data EXISTS")
    for d in sorted(env.iterdir()):
        if d.is_dir():
            count = len(list(d.rglob("*")))
            print(f"    Env-Data/{d.name}/  ({count} files)")
else:
    print("  Env-Data NOT found at TrainData/Env-Data")

print()
print("[7] Data3D basin folders at root (never read by our LSTM)")
print("-" * 50)
for basin in BASINS:
    p = ROOT / basin
    if p.exists():
        subdirs = [d.name for d in sorted(p.iterdir()) if d.is_dir()][:4]
        print(f"  TrainData/{basin}/  EXISTS  (sample subdirs: {subdirs})")
    else:
        print(f"  TrainData/{basin}/  not present")

[6] Env-Data
--------------------------------------------------
  Env-Data EXISTS
    Env-Data/EP/  (16496 files)
    Env-Data/NA/  (18955 files)
    Env-Data/NI/  (1230 files)
    Env-Data/SI/  (20343 files)
    Env-Data/SP/  (9859 files)
    Env-Data/WP/  (42498 files)

[7] Data3D basin folders at root (never read by our LSTM)
--------------------------------------------------
  TrainData/EP/  EXISTS  (sample subdirs: ['1988', '1989', '1990', '1991'])
  TrainData/NA/  EXISTS  (sample subdirs: ['1960', '1961', '1962', '1963'])
  TrainData/NI/  EXISTS  (sample subdirs: ['1992', '1997', '2005', '2006'])
  TrainData/SI/  EXISTS  (sample subdirs: ['1973', '1974', '1975', '1976'])
  TrainData/SP/  EXISTS  (sample subdirs: ['1970', '1971', '1972', '1973'])
  TrainData/WP/  EXISTS  (sample subdirs: ['1950', '1951', '1952', '1953'])
